# Object Detection using YOLOv11 on IDD Dataset

This notebook demonstrates how to train a YOLOv11 model on the Indian Driving Dataset.

In [ ]:
! nvidia-smi -L

In [ ]:
%%time
! pip install ultralytics

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import glob
import random
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

import IPython.display as display
from PIL import Image
import cv2

from ultralytics import YOLO

In [ ]:
! wandb disabled

In [ ]:
class CFG:
    DEBUG = False
    FRACTION = 0.10 if DEBUG else 1.0
    SEED = 88
    
    # classes
    CLASSES = [
        'animal', 'autorickshaw', 'bicycle', 'bus', 'car', 'caravan', 
        'motorcycle', 'person', 'rider', 'traffic light', 'traffic sign', 
        'trailer', 'train', 'truck', 'vehicle fallback'
    ]
    
    NUM_CLASSES_TO_TRAIN = len(CLASSES)
    
    # training
    EPOCHS = 3 if DEBUG else 100 
    BATCH_SIZE = 16
    BASE_MODEL = 'yolo11n'
    EXP_NAME = f'ppe_css_{EPOCHS}_epochs'
    
    OPTIMIZER = 'auto'
    LR = 1e-3
    LR_FACTOR = 0.01
    WEIGHT_DECAY = 5e-4
    DROPOUT = 0.025
    PATIENCE = 25
    PROFILE = False
    LABEL_SMOOTHING = 0.0
    
    # paths
    CUSTOM_DATASET_DIR = '/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset'
    OUTPUT_DIR = '.'

In [ ]:
dict_file = {
    'train': os.path.join(CFG.CUSTOM_DATASET_DIR, 'train'),
    'val': os.path.join(CFG.CUSTOM_DATASET_DIR, 'val'),
    'test': os.path.join(CFG.CUSTOM_DATASET_DIR, 'test'),
    'nc': CFG.NUM_CLASSES_TO_TRAIN,
    'names': CFG.CLASSES
}

with open(os.path.join(CFG.OUTPUT_DIR, 'data.yaml'), 'w+') as file:
    yaml.dump(dict_file, file)

In [ ]:
def read_yaml_file(file_path = CFG.CUSTOM_DATASET_DIR):
    with open(file_path, 'r') as file:
        try:
            data = yaml.safe_load(file)
            return data
        except yaml.YAMLError as e:
            print("Error reading YAML:", e)
            return None

def print_yaml_data(data):
    formatted_yaml = yaml.dump(data, default_style=False)
    print(formatted_yaml)

file_path = os.path.join(CFG.OUTPUT_DIR, 'data.yaml')
yaml_data = read_yaml_file(file_path)
if yaml_data:
    print_yaml_data(yaml_data)

In [ ]:
def display_image(image, print_info = True, hide_axis = False):
    if isinstance(image, str):
        img = Image.open(image)
        plt.imshow(img)
    elif isinstance(image, np.ndarray):
        image = image[..., ::-1] # BGR to RGB
        img = Image.fromarray(image)
        plt.imshow(img)
    else:
        raise ValueError("Unsupported image format")
        
    if print_info:
        print('Type: ', type(img), '\n')
        print('Shape: ', np.array(img).shape, '\n')
    
    if hide_axis:
        plt.axis('off')
    plt.show()

In [ ]:
example_image_path = '/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/train/images/15-07-18-upload_0000525.jpg'
display_image(example_image_path, print_info = True, hide_axis = False)

In [ ]:
def plot_random_images(folder_path, num_images = 10):
    image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    selected_files = random.sample(image_files, num_images)
    num_cols = 5
    num_rows = (num_images + num_cols - 1) // num_cols
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8))
    
    for i, file_name in enumerate(selected_files):
        img = Image.open(os.path.join(folder_path, file_name))
        ax = axes[i // num_cols, i % num_cols] if num_rows > 1 else axes[i % num_cols]
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
folder_path = CFG.CUSTOM_DATASET_DIR + '/train/images/'
plot_random_images(folder_path, num_images=20)

In [ ]:
class_idx = {str(i): cls for i, cls in enumerate(CFG.CLASSES)}
class_info = []
class_stat = {}

for mode in ['train', 'val', 'test']:
    path = os.path.join(CFG.CUSTOM_DATASET_DIR, mode, 'labels')
    class_count = {cls: 0 for cls in CFG.CLASSES}
    for file in os.listdir(path):
        with open(os.path.join(path, file)) as f:
            lines = f.readlines()
            for line in lines:
                class_count[class_idx[line.split()[0]]] += 1
    class_stat[mode] = class_count
    class_info.append({'Mode': mode, **class_count, 'Data_Volume': len(os.listdir(path))})

dataset_stats_df = pd.DataFrame(class_info)
print(dataset_stats_df)

In [ ]:
model = YOLO(CFG.BASE_MODEL + '.pt')
model.train(
    data=os.path.join(CFG.OUTPUT_DIR, 'data.yaml'),
    epochs=CFG.EPOCHS,
    batch=CFG.BATCH_SIZE,
    imgsz=640,
    optimizer=CFG.OPTIMIZER,
    lr0=CFG.LR,
    lrf=CFG.LR_FACTOR,
    weight_decay=CFG.WEIGHT_DECAY,
    dropout=CFG.DROPOUT,
    fraction=CFG.FRACTION,
    patience=CFG.PATIENCE,
    name=f'{CFG.BASE_MODEL}_{CFG.EXP_NAME}',
    seed=CFG.SEED,
    device=[0, 1] if not CFG.DEBUG else 0,
    val=True
)

In [ ]:
model_path = '/kaggle/working/runs/detect/ppe_css_100_epochs/weights/best.pt'
test_images_path = '/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/test/images'
results = model.predict(source=test_images_path, conf=0.25, save=True)